# Time profiling: DQN on LunarLander (CPU)

Run the 3 cells in order on a **CPU runtime** (the whole thing takes about 5-8 minutes). Then copy the block printed by the last cell and send it back.

In [ ]:
# 1) install (same as your gamma-sweep notebook)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'swig'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium[box2d]', 'tf_keras'], check=True)
print('ok')

ok


In [ ]:
%%writefile profile_dqn.py
#!/usr/bin/env python
"""
Time profiling for the DQN / LunarLander code (gamma sweep notebook).

Usage (Colab):   !python profile_dqn.py all
Optional:        !python profile_dqn.py all --steps 5000      (faster, noisier)

It measures, on THIS machine:
  1. cost of each component per call (env.step, act, buffer, learn, ...)
  2. a breakdown of the real training loop (where does each step's time go?)
  3. what a few cheap fixes would give (numpy act, XLA, 1 thread per process)
  4. how well running several processes in parallel scales on this machine
  5. (only if a GPU runtime is active) CPU vs GPU for the training step
Everything is printed at the end in one block + saved to profile_report.json.
"""
import os, sys, time, json, argparse, subprocess, platform, itertools
import numpy as np

PC = time.perf_counter
ENV_ID = "LunarLander-v3"
GAMMA, LR, TAU, HIDDEN, BATCH, UPDATE_EVERY = 0.99, 1e-3, 1e-3, 64, 64, 4
SWEEP_TOTAL_STEPS = 3_745_325      # total env steps of your real sweep (sum over the 8 runs)
SWEEP_REAL_MINUTES = 75.6          # what that sweep actually took

tf = None
gym = None


def emit(tag, obj):
    print("RESULT " + tag + " " + json.dumps(obj), flush=True)


# --------------------------------------------------------------------------- setup (workers only)
def setup_tf(threads):
    """Import TF/gym. Thread settings must be applied before TF runs any op."""
    global tf, gym
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    import tensorflow as _tf
    import gymnasium as _gym
    tf, gym = _tf, _gym
    if threads > 0:
        tf.config.threading.set_intra_op_parallelism_threads(threads)
        tf.config.threading.set_inter_op_parallelism_threads(threads)


# --------------------------------------------------------------------------- your code (copied as is)
class ReplayBuffer:
    def __init__(self, capacity, state_dim, rng):
        self.capacity, self.rng = capacity, rng
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int32)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.terminals = np.zeros(capacity, dtype=np.float32)
        self._next_idx, self.size = 0, 0

    def add(self, state, action, reward, next_state, terminal):
        i = self._next_idx
        self.states[i] = state
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_states[i] = next_state
        self.terminals[i] = float(terminal)
        self._next_idx = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (self.states[idx], self.actions[idx], self.rewards[idx],
                self.next_states[idx], self.terminals[idx])


def build_qnet(state_dim, n_actions, hidden_units):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(state_dim,)),
        tf.keras.layers.Dense(hidden_units, activation="relu"),
        tf.keras.layers.Dense(hidden_units, activation="relu"),
        tf.keras.layers.Dense(n_actions, activation="linear"),
    ])


class DQNAgent:
    def __init__(self, state_dim=8, n_actions=4, gamma=GAMMA, seed=0, xla=False):
        self.n_actions = n_actions
        self.tau = TAU
        self.gamma = tf.constant(gamma, dtype=tf.float32)
        self.rng = np.random.default_rng(seed)
        self.q = build_qnet(state_dim, n_actions, HIDDEN)
        self.target = build_qnet(state_dim, n_actions, HIDDEN)
        self.target.set_weights(self.q.get_weights())
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
        self._greedy = tf.function(self._greedy_impl)
        self._train = tf.function(self._train_impl, jit_compile=xla)

    def _greedy_impl(self, state):
        q_values = self.q(tf.expand_dims(state, 0), training=False)
        return tf.argmax(q_values[0])

    def act(self, state, epsilon):
        if epsilon > 0.0 and self.rng.random() < epsilon:
            return int(self.rng.integers(self.n_actions))
        return int(self._greedy(tf.convert_to_tensor(state, dtype=tf.float32)))

    def _train_impl(self, states, actions, rewards, next_states, terminals):
        next_q = tf.reduce_max(self.target(next_states, training=False), axis=1)
        targets = rewards + self.gamma * next_q * (1.0 - terminals)
        with tf.GradientTape() as tape:
            q_all = self.q(states, training=True)
            rows = tf.range(tf.shape(actions)[0])
            q_taken = tf.gather_nd(q_all, tf.stack([rows, actions], axis=1))
            loss = tf.reduce_mean(tf.square(tf.stop_gradient(targets) - q_taken))
        grads = tape.gradient(loss, self.q.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.q.trainable_variables))
        for w_t, w in zip(self.target.trainable_variables, self.q.trainable_variables):
            w_t.assign(self.tau * w + (1.0 - self.tau) * w_t)
        return loss

    def learn(self, batch):
        return self._train(*batch)


# --------------------------------------------------------------------------- helpers
def np_greedy(W, s):
    """Same 8-64-64-4 ReLU network, plain numpy. W = [W1, b1, W2, b2, W3, b3]."""
    h = np.maximum(s @ W[0] + W[1], 0.0)
    h = np.maximum(h @ W[2] + W[3], 0.0)
    return int(np.argmax(h @ W[4] + W[5]))


def get_np_weights(agent):
    return [v.numpy() for v in agent.q.trainable_variables]


def timeit(fn, n, warmup=30):
    for _ in range(warmup):
        fn()
    t0 = PC()
    for _ in range(n):
        fn()
    return (PC() - t0) / n * 1e6          # microseconds per call


def collect_transitions(env, n, seed=0):
    rng = np.random.default_rng(seed)
    s, _ = env.reset(seed=seed)
    out = []
    while len(out) < n:
        a = int(rng.integers(4))
        ns, r, te, tr, _ = env.step(a)
        out.append((s, a, r, ns, te))
        s = ns
        if te or tr:
            s, _ = env.reset()
    return out


def device_name(device):
    return "/GPU:0" if device == "gpu" else "/CPU:0"


# --------------------------------------------------------------------------- worker: micro-benchmarks
def run_micro(device, threads):
    with tf.device(device_name(device)):
        return _micro(device, threads)


def _micro(device, threads):
    res = {"device": device, "threads": threads,
           "gpu_available": bool(tf.config.list_physical_devices("GPU")),
           "tf_version": tf.__version__, "gym_version": gym.__version__}
    env = gym.make(ENV_ID)

    # --- 1. environment alone (random actions)
    n = 5000
    acts = np.random.default_rng(0).integers(0, 4, size=n)
    env.reset(seed=0)
    t_step = t_reset = 0.0
    n_reset = 0
    for a in acts:
        t0 = PC()
        _, _, te, tr, _ = env.step(int(a))
        t_step += PC() - t0
        if te or tr:
            t0 = PC()
            env.reset()
            t_reset += PC() - t0
            n_reset += 1
    res["env_step_us"] = t_step / n * 1e6
    res["env_reset_us"] = t_reset / max(n_reset, 1) * 1e6

    trans = collect_transitions(env, 6000)
    states = np.array([t[0] for t in trans], dtype=np.float32)
    agent = DQNAgent()
    buf = ReplayBuffer(100_000, 8, np.random.default_rng(1))
    for t in trans:
        buf.add(*t)

    # --- 2. acting
    it = itertools.cycle(states)
    res["act_tf_greedy_us"] = timeit(lambda: agent.act(next(it), 0.0), 2000)
    res["act_random_us"] = timeit(lambda: agent.act(next(it), 1.0), 20000)
    W = get_np_weights(agent)
    res["act_numpy_us"] = timeit(lambda: np_greedy(W, next(it)), 20000)
    same = np.mean([np_greedy(W, s) == agent.act(s, 0.0) for s in states[:300]])
    res["act_numpy_agreement"] = float(same)

    # batched acting (what a vectorised env would use): cost per CALL for B states
    for B in (1, 8, 32):
        xb = states[:B]
        f = tf.function(lambda x: tf.argmax(agent.q(x, training=False), axis=1))
        res[f"act_tf_batch{B}_us_per_call"] = timeit(lambda: f(xb).numpy(), 1000)
        res[f"act_numpy_batch{B}_us_per_call"] = timeit(
            lambda: np.argmax(np.maximum(np.maximum(xb @ W[0] + W[1], 0) @ W[2] + W[3], 0) @ W[4] + W[5], axis=1), 5000)

    # --- 3. buffer
    tit = itertools.cycle(trans)
    res["buffer_add_us"] = timeit(lambda: buf.add(*next(tit)), 20000)
    res["buffer_sample64_us"] = timeit(lambda: buf.sample(64), 5000)

    # --- 4. learning step (batch pre-sampled -> pure train step incl. numpy->tensor conversion)
    for bs in (64, 256, 1024):
        batch = buf.sample(bs)
        res[f"learn_bs{bs}_us"] = timeit(lambda: agent.learn(batch).numpy(), 300 if bs < 1024 else 150)
    res["sample_plus_learn64_us"] = timeit(lambda: agent.learn(buf.sample(64)).numpy(), 500)

    # --- 5. XLA-compiled training step (may not work on every setup)
    try:
        ax = DQNAgent(xla=True)
        for bs in (64, 1024):
            batch = buf.sample(bs)
            res[f"learn_bs{bs}_xla_us"] = timeit(lambda: ax.learn(batch).numpy(), 300 if bs < 1024 else 150)
    except Exception as e:
        res["xla_error"] = (type(e).__name__ + ": " + str(e))[:300]
    return res


# --------------------------------------------------------------------------- worker: real-loop breakdown
def run_loop(eps, steps, device, threads, use_np, timers, start_at=None):
    with tf.device(device_name(device)):
        return _loop(eps, steps, threads, use_np, timers, start_at)


def _loop(eps, steps, threads, use_np, timers, start_at):
    env = gym.make(ENV_ID)
    trans = collect_transitions(env, 6000)
    agent = DQNAgent()
    buf = ReplayBuffer(100_000, 8, np.random.default_rng(1))
    for t in trans:
        buf.add(*t)
    for _ in range(20):                       # trace / warm up
        agent.learn(buf.sample(BATCH))
        agent.act(trans[0][0], 0.0)
    rng = np.random.default_rng(0)
    W = get_np_weights(agent) if use_np else None
    state, _ = env.reset(seed=1)
    acc = dict(act=0.0, env_step=0.0, env_reset=0.0, buf_add=0.0, buf_sample=0.0, learn=0.0, np_sync=0.0)
    n_learn = n_ep = 0
    if start_at:                              # line up several processes on the same wall-clock instant
        while time.time() < start_at:
            time.sleep(0.01)
    t_start = PC()
    for step in range(1, steps + 1):
        if timers:
            t0 = PC()
        if eps > 0.0 and rng.random() < eps:
            a = int(rng.integers(4))
        elif use_np:
            a = np_greedy(W, state)
        else:
            a = int(agent._greedy(tf.convert_to_tensor(state, dtype=tf.float32)))
        if timers:
            t1 = PC(); acc["act"] += t1 - t0
        ns, r, te, tr, _ = env.step(a)
        if timers:
            t2 = PC(); acc["env_step"] += t2 - t1
        buf.add(state, a, r, ns, te)
        if timers:
            t3 = PC(); acc["buf_add"] += t3 - t2
        state = ns
        if te or tr:
            if timers:
                t4 = PC()
            state, _ = env.reset()
            n_ep += 1
            if timers:
                acc["env_reset"] += PC() - t4
        if step % UPDATE_EVERY == 0:
            if timers:
                t5 = PC()
            batch = buf.sample(BATCH)
            if timers:
                t6 = PC(); acc["buf_sample"] += t6 - t5
            agent.learn(batch)
            n_learn += 1
            if timers:
                t7 = PC(); acc["learn"] += t7 - t6
            if use_np:
                W = get_np_weights(agent)
                if timers:
                    acc["np_sync"] += PC() - t7
    total = PC() - t_start
    out = {"eps": eps, "steps": steps, "use_np": use_np, "threads": threads, "timers": timers,
           "steps_per_s": steps / total, "seconds": total, "episodes": n_ep, "learn_calls": n_learn}
    if timers:
        per = {k: v / steps * 1e6 for k, v in acc.items()}      # microseconds per ENV STEP
        per["other_overhead"] = total / steps * 1e6 - sum(per.values())
        out["us_per_step"] = per
    return out


# --------------------------------------------------------------------------- orchestrator (no TF import here)
def run_worker(args_list, timeout=1800):
    cmd = [sys.executable, os.path.abspath(__file__)] + args_list
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return parse(p.stdout), p


def parse(text):
    out = []
    for line in text.splitlines():
        if line.startswith("RESULT "):
            _, tag, payload = line.split(" ", 2)
            out.append((tag, json.loads(payload)))
    return out


def run_parallel(k, extra, start_delay=30):
    start_at = time.time() + start_delay
    cmd = [sys.executable, os.path.abspath(__file__), "loop"] + extra + ["--start-at", str(start_at)]
    procs = [subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) for _ in range(k)]
    results = []
    for p in procs:
        so, se = p.communicate()
        results += [r for _, r in parse(so)]
    return results


def orchestrate(steps):
    ncpu = os.cpu_count() or 1
    report = {"system": {"cpu_count": ncpu, "platform": platform.platform(), "python": platform.python_version()}}
    try:
        with open("/proc/cpuinfo") as f:
            report["system"]["cpu_model"] = next(l.split(":", 1)[1].strip() for l in f if l.startswith("model name"))
    except Exception:
        pass

    print(f"[1/5] component micro-benchmarks (CPU) ...", flush=True)
    res, p = run_worker(["micro", "--device", "cpu"])
    if not res:
        print("micro failed:\n", p.stderr[-3000:]); sys.exit(1)
    report["micro_cpu"] = res[0][1]

    print(f"[2/5] real training loop breakdown ({steps} steps each) ...", flush=True)
    loops = {}
    for name, eps, np_flag in [("tf_eps1.0", 1.0, False), ("tf_eps0.05", 0.05, False), ("numpy_act_eps0.05", 0.05, True)]:
        args = ["loop", "--eps", str(eps), "--steps", str(steps), "--mode", "both"] + (["--np-act"] if np_flag else [])
        res, p = run_worker(args)
        if not res:
            print(name, "failed:\n", p.stderr[-2000:]); continue
        loops[name] = {tag: r for tag, r in res}
    report["loops"] = loops

    print("[3/5] parallel scaling (processes running at the same time) ...", flush=True)
    par = []
    configs = [("1 proc, default threads", 1, 0, False), ("1 proc, 1 thread", 1, 1, False)]
    if ncpu > 1:
        configs += [(f"{ncpu} procs, default threads", ncpu, 0, False),
                    (f"{ncpu} procs, 1 thread each", ncpu, 1, False),
                    (f"{ncpu} procs, 1 thread each, numpy act", ncpu, 1, True)]
    for label, k, th, np_flag in configs:
        extra = ["--eps", "0.05", "--steps", str(steps), "--mode", "plain", "--threads", str(th)] + (["--np-act"] if np_flag else [])
        if k == 1:
            res, p = run_worker(["loop"] + extra)
            rs = [r for _, r in res]
        else:
            rs = run_parallel(k, extra)
        if rs:
            par.append({"label": label, "procs": k, "per_proc_steps_per_s": [round(r["steps_per_s"]) for r in rs],
                        "total_steps_per_s": round(sum(r["steps_per_s"] for r in rs))})
        else:
            print(label, "failed")
    report["parallel"] = par

    gpu = report["micro_cpu"].get("gpu_available")
    if gpu:
        print("[4/5] GPU micro-benchmarks ...", flush=True)
        res, p = run_worker(["micro", "--device", "gpu"])
        if res:
            report["micro_gpu"] = res[0][1]
    else:
        print("[4/5] no GPU visible -> skipped (switch runtime to GPU and run `!python profile_dqn.py micro --device gpu` if you want it)", flush=True)

    print("[5/5] building report ...\n", flush=True)
    with open("profile_report.json", "w") as f:
        json.dump(report, f, indent=1)
    print_report(report)


def print_report(R):
    m = R["micro_cpu"]
    print("=" * 70)
    print("COPY EVERYTHING BETWEEN THE LINES AND SEND IT BACK")
    print("=" * 70)
    print("SYSTEM:", json.dumps(R["system"]))
    print(f"tensorflow {m['tf_version']} | gymnasium {m['gym_version']} | GPU visible: {m['gpu_available']}")

    print("\n[A] cost per call (microseconds, CPU)")
    keys = ["env_step_us", "env_reset_us", "act_tf_greedy_us", "act_random_us", "act_numpy_us", "act_numpy_agreement",
            "buffer_add_us", "buffer_sample64_us", "learn_bs64_us", "sample_plus_learn64_us",
            "learn_bs256_us", "learn_bs1024_us", "learn_bs64_xla_us", "learn_bs1024_xla_us", "xla_error",
            "act_tf_batch1_us_per_call", "act_tf_batch8_us_per_call", "act_tf_batch32_us_per_call",
            "act_numpy_batch1_us_per_call", "act_numpy_batch8_us_per_call", "act_numpy_batch32_us_per_call"]
    for k in keys:
        if k in m:
            v = m[k]
            print(f"  {k:34s} {v:10.1f}" if isinstance(v, (int, float)) else f"  {k:34s} {v}")

    print("\n[B] real training loop: microseconds per ENV STEP (share of total)")
    for name, d in R["loops"].items():
        t = d.get("timed")
        p = d.get("plain")
        if not t:
            continue
        tot = 1e6 / t["steps_per_s"]
        print(f"  --- {name}:  {p['steps_per_s'] if p else t['steps_per_s']:.0f} steps/s (plain run) | "
              f"{t['steps_per_s']:.0f} steps/s (with timers) | episodes in run: {t['episodes']}")
        for k, v in t["us_per_step"].items():
            print(f"      {k:15s} {v:8.1f} us   {100 * v / tot:5.1f}%")

    print("\n[C] parallel scaling (eps=0.05)")
    for d in R["parallel"]:
        print(f"  {d['label']:42s} total {d['total_steps_per_s']:6d} steps/s   per-proc {d['per_proc_steps_per_s']}")

    if "micro_gpu" in R:
        g = R["micro_gpu"]
        print("\n[D] training step, CPU vs GPU (microseconds per call)")
        for bs in (64, 256, 1024):
            print(f"  batch {bs:5d}:  CPU {m[f'learn_bs{bs}_us']:9.1f}   GPU {g[f'learn_bs{bs}_us']:9.1f}")
        print(f"  act (1 state): CPU {m['act_tf_greedy_us']:.1f}   GPU {g['act_tf_greedy_us']:.1f}")

    print("\n[E] sanity check: does the profile explain your real sweep?")
    for name in ("tf_eps1.0", "tf_eps0.05"):
        d = R["loops"].get(name, {}).get("plain")
        if d:
            mins = SWEEP_TOTAL_STEPS / d["steps_per_s"] / 60
            print(f"  predicted sweep time at {name}: {mins:5.1f} min   (real sweep: {SWEEP_REAL_MINUTES} min)")
    a = R["loops"].get("tf_eps0.05", {}).get("plain")
    b = R["loops"].get("numpy_act_eps0.05", {}).get("plain")
    if a and b:
        print(f"  numpy-act speed-up (1 process, eps 0.05): x{b['steps_per_s'] / a['steps_per_s']:.2f}")
    print("=" * 70)
    print("full numbers also saved in profile_report.json")


# --------------------------------------------------------------------------- entry point
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["all", "micro", "loop"])
    ap.add_argument("--steps", type=int, default=10000)
    ap.add_argument("--eps", type=float, default=0.05)
    ap.add_argument("--device", default="cpu", choices=["cpu", "gpu"])
    ap.add_argument("--threads", type=int, default=0, help="0 = TensorFlow default")
    ap.add_argument("--np-act", action="store_true")
    ap.add_argument("--mode", default="both", choices=["both", "timed", "plain"])
    ap.add_argument("--start-at", type=float, default=None)
    a = ap.parse_args()

    if a.cmd == "all":
        orchestrate(a.steps)
        return
    setup_tf(a.threads)
    if a.cmd == "micro":
        emit("micro", run_micro(a.device, a.threads))
    else:
        if a.mode in ("both", "timed"):
            emit("timed", run_loop(a.eps, a.steps, a.device, a.threads, a.np_act, True))
        if a.mode in ("both", "plain"):
            emit("plain", run_loop(a.eps, a.steps, a.device, a.threads, a.np_act, False, a.start_at))


if __name__ == "__main__":
    main()


Writing profile_dqn.py


In [ ]:
# 3) run everything (restart the runtime first if TensorFlow was already imported in this session)
!python profile_dqn.py all

[1/5] component micro-benchmarks (CPU) ...
[2/5] real training loop breakdown (10000 steps each) ...
[3/5] parallel scaling (processes running at the same time) ...
[4/5] no GPU visible -> skipped (switch runtime to GPU and run `!python profile_dqn.py micro --device gpu` if you want it)
[5/5] building report ...

COPY EVERYTHING BETWEEN THE LINES AND SEND IT BACK
SYSTEM: {"cpu_count": 2, "platform": "Linux-6.6.122+-x86_64-with-glibc2.39", "python": "3.13.16", "cpu_model": "Intel(R) Xeon(R) CPU @ 2.20GHz"}
tensorflow 2.21.0 | gymnasium 1.3.0 | GPU visible: False

[A] cost per call (microseconds, CPU)
  env_step_us                              49.6
  env_reset_us                            405.2
  act_tf_greedy_us                        643.3
  act_random_us                             3.2
  act_numpy_us                             15.0
  act_numpy_agreement                       1.0
  buffer_add_us                             3.5
  buffer_sample64_us                       23.9
  learn_b

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','swig'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','gymnasium[box2d]','tf_keras'],check=True)

SCRIPT = r'''#!/usr/bin/env python
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # tiny matrices: BLAS threads only add overhead
import sys, time, json, argparse, subprocess, platform, itertools
import numpy as np

PC = time.perf_counter
ENV_ID = "LunarLander-v3"
BATCH, UPDATE_EVERY = 64, 4
SWEEP_TOTAL_STEPS = 3_745_325
SWEEP_REAL_MINUTES = 75.6
tf = None
gym = None


def emit(tag, obj):
    print("RESULT " + tag + " " + json.dumps(obj), flush=True)


def setup(need_tf):
    global tf, gym
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    import gymnasium as _gym
    gym = _gym
    if need_tf:
        import tensorflow as _tf
        tf = _tf
        tf.config.threading.set_intra_op_parallelism_threads(1)
        tf.config.threading.set_inter_op_parallelism_threads(1)


# ===================== numpy DQN (verified against finite differences)
GAMMA, LR, TAU = 0.99, 1e-3, 1e-3

def glorot(rng, shapes=((8,64),(64,64),(64,4))):
    W=[]
    for fi,fo in shapes:
        lim=np.sqrt(6.0/(fi+fo))
        W += [rng.uniform(-lim,lim,size=(fi,fo)).astype(np.float32), np.zeros(fo,np.float32)]
    return W

class NumpyDQN:
    """Whole DQN (8-64-64-4 ReLU, Adam, soft target update) in plain numpy. The acting weights ARE self.p."""
    def __init__(self, W0, gamma=GAMMA, seed=0):
        self.p=[w.copy() for w in W0]; self.t=[w.copy() for w in W0]
        self.m=[np.zeros_like(w) for w in W0]; self.v=[np.zeros_like(w) for w in W0]
        self.n=0; self.gamma=np.float32(gamma); self.rng=np.random.default_rng(seed)

    @staticmethod
    def _fwd(P, x):
        h1=np.maximum(x@P[0]+P[1],0.0); h2=np.maximum(h1@P[2]+P[3],0.0)
        return h1,h2,h2@P[4]+P[5]

    def act(self, s, eps):
        if eps>0.0 and self.rng.random()<eps: return int(self.rng.integers(4))
        return int(np.argmax(self._fwd(self.p, s)[2]))

    def grads(self, batch):
        s,a,r,ns,term=batch
        y = r + self.gamma*self._fwd(self.t, ns)[2].max(axis=1)*(1.0-term)
        h1,h2,q = self._fwd(self.p, s)
        B=s.shape[0]; idx=np.arange(B)
        err = q[idx,a]-y
        loss=float(np.mean(err*err))
        dq=np.zeros_like(q); dq[idx,a]=2.0*err/B
        dW3=h2.T@dq; db3=dq.sum(0)
        dh2=(dq@self.p[4].T)*(h2>0)
        dW2=h1.T@dh2; db2=dh2.sum(0)
        dh1=(dh2@self.p[2].T)*(h1>0)
        dW1=s.T@dh1; db1=dh1.sum(0)
        return loss,[dW1,db1,dW2,db2,dW3,db3]

    def learn(self, batch):
        loss,g=self.grads(batch)
        self.n+=1; b1,b2,eps=0.9,0.999,1e-7
        lr_t=LR*np.sqrt(1-b2**self.n)/(1-b1**self.n)
        for p,m,v,gr,t in zip(self.p,self.m,self.v,g,self.t):
            m*=b1; m+=(1-b1)*gr
            v*=b2; v+=(1-b2)*gr*gr
            p-=lr_t*m/(np.sqrt(v)+eps)
            t*=(1-TAU); t+=TAU*p
        return loss


# ===================== replay buffer (your code)
class ReplayBuffer:
    def __init__(self, capacity, state_dim, rng):
        self.capacity, self.rng = capacity, rng
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int32)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.terminals = np.zeros(capacity, dtype=np.float32)
        self._next_idx, self.size = 0, 0

    def add(self, state, action, reward, next_state, terminal):
        i = self._next_idx
        self.states[i] = state; self.actions[i] = action; self.rewards[i] = reward
        self.next_states[i] = next_state; self.terminals[i] = float(terminal)
        self._next_idx = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (self.states[idx], self.actions[idx], self.rewards[idx], self.next_states[idx], self.terminals[idx])


# ===================== TensorFlow variants
def build_qnet():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(8,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(4, activation="linear")])


class TFBaseline:
    """Your current agent (act via tf.function, learn via Keras Adam)."""
    def __init__(self, W0=None, gamma=GAMMA, seed=0):
        self.gamma = tf.constant(gamma, dtype=tf.float32)
        self.rng = np.random.default_rng(seed)
        self.q, self.target = build_qnet(), build_qnet()
        if W0 is not None:
            self.q.set_weights(W0)
        self.target.set_weights(self.q.get_weights())
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
        self._greedy = tf.function(self._greedy_impl)
        self._train = tf.function(self._train_impl)

    def _greedy_impl(self, state):
        return tf.argmax(self.q(tf.expand_dims(state, 0), training=False)[0])

    def act(self, state, eps):
        if eps > 0.0 and self.rng.random() < eps:
            return int(self.rng.integers(4))
        return int(self._greedy(tf.convert_to_tensor(state, dtype=tf.float32)))

    def _train_impl(self, states, actions, rewards, next_states, terminals):
        next_q = tf.reduce_max(self.target(next_states, training=False), axis=1)
        targets = rewards + self.gamma * next_q * (1.0 - terminals)
        with tf.GradientTape() as tape:
            q_all = self.q(states, training=True)
            rows = tf.range(tf.shape(actions)[0])
            q_taken = tf.gather_nd(q_all, tf.stack([rows, actions], axis=1))
            loss = tf.reduce_mean(tf.square(tf.stop_gradient(targets) - q_taken))
        grads = tape.gradient(loss, self.q.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.q.trainable_variables))
        for w_t, w in zip(self.target.trainable_variables, self.q.trainable_variables):
            w_t.assign(TAU * w + (1.0 - TAU) * w_t)
        return loss

    def learn(self, batch):
        return self._train(*batch)


class TFNpAct(TFBaseline):
    """TF learn + numpy act. After each update ALL weights come back in ONE transfer (concat),
    instead of 6 separate .numpy() calls."""
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self._shapes = [tuple(v.shape) for v in self.q.trainable_variables]
        self._sizes = [int(np.prod(s)) for s in self._shapes]
        self._train_flat = tf.function(self._train_flat_impl)
        self._unflatten()
        self.W = self._split(np.concatenate([v.numpy().ravel() for v in self.q.trainable_variables]))

    def _train_flat_impl(self, *batch):
        loss = self._train_impl(*batch)
        return tf.concat([tf.reshape(v, [-1]) for v in self.q.trainable_variables], 0), loss

    def _unflatten(self):
        pass

    def _split(self, flat):
        out, o = [], 0
        for shp, n in zip(self._shapes, self._sizes):
            out.append(flat[o:o + n].reshape(shp)); o += n
        return out

    def act(self, state, eps):
        if eps > 0.0 and self.rng.random() < eps:
            return int(self.rng.integers(4))
        W = self.W
        h = np.maximum(state @ W[0] + W[1], 0.0)
        h = np.maximum(h @ W[2] + W[3], 0.0)
        return int(np.argmax(h @ W[4] + W[5]))

    def learn(self, batch):
        flat, loss = self._train_flat(*batch)
        self.W = self._split(flat.numpy())
        return loss


class TFManual:
    """TF with raw variables + hand-written Adam (no Keras layers/optimizer). K updates per call."""
    def __init__(self, W0, K=1):
        self.K = K
        mk = lambda ws: [tf.Variable(w) for w in ws]
        self.p, self.t = mk(W0), mk(W0)
        self.m = mk([np.zeros_like(w) for w in W0]); self.v = mk([np.zeros_like(w) for w in W0])
        self.n = tf.Variable(0.0)
        self.train = tf.function(self._train)

    @staticmethod
    def _fwd(P, x):
        h = tf.nn.relu(tf.matmul(x, P[0]) + P[1])
        h = tf.nn.relu(tf.matmul(h, P[2]) + P[3])
        return tf.matmul(h, P[4]) + P[5]

    def _one(self, s, a, r, ns, term):
        y = r + GAMMA * tf.reduce_max(self._fwd(self.t, ns), axis=1) * (1.0 - term)
        with tf.GradientTape() as tape:
            q = tf.reduce_sum(self._fwd(self.p, s) * tf.one_hot(a, 4), axis=1)
            loss = tf.reduce_mean(tf.square(tf.stop_gradient(y) - q))
        g = tape.gradient(loss, self.p)
        self.n.assign_add(1.0)
        lr_t = LR * tf.sqrt(1.0 - 0.999 ** self.n) / (1.0 - 0.9 ** self.n)
        for p, m, v, gr in zip(self.p, self.m, self.v, g):
            m.assign(0.9 * m + 0.1 * gr)
            v.assign(0.999 * v + 0.001 * gr * gr)
            p.assign_sub(lr_t * m / (tf.sqrt(v) + 1e-7))
        for t, p in zip(self.t, self.p):
            t.assign(TAU * p + (1.0 - TAU) * t)
        return loss

    def _train(self, S, A, R, NS, T):          # arrays have a leading dimension K
        loss = 0.0
        for i in range(self.K):
            loss = self._one(S[i], A[i], R[i], NS[i], T[i])
        return loss


# ===================== helpers
def timeit(fn, n, warmup=30):
    for _ in range(warmup):
        fn()
    t0 = PC()
    for _ in range(n):
        fn()
    return (PC() - t0) / n * 1e6


def collect_transitions(env, n, seed=0):
    rng = np.random.default_rng(seed)
    s, _ = env.reset(seed=seed)
    out = []
    while len(out) < n:
        a = int(rng.integers(4))
        ns, r, te, tr, _ = env.step(a)
        out.append((s, a, r, ns, te))
        s = ns
        if te or tr:
            s, _ = env.reset()
    return out


def safe(res, name, fn):
    try:
        res[name] = fn()
    except Exception as e:
        res[name + "_error"] = (type(e).__name__ + ": " + str(e))[:300]


def stack_batches(buf, K):
    bs = [buf.sample(BATCH) for _ in range(K)]
    return tuple(np.stack([b[i] for b in bs]) for i in range(5))


# ===================== micro-benchmarks
def run_micro():
    res = {"tf_version": tf.__version__, "gym_version": gym.__version__}
    env = gym.make(ENV_ID)
    trans = collect_transitions(env, 6000)
    buf = ReplayBuffer(100_000, 8, np.random.default_rng(1))
    for t in trans:
        buf.add(*t)
    batch = buf.sample(BATCH)
    W0 = glorot(np.random.default_rng(0))
    states = np.array([t[0] for t in trans], dtype=np.float32)
    it = itertools.cycle(states)

    # --- correctness: numpy gradients == TF gradients (same weights, same batch)
    def check():
        ag = TFBaseline(W0)
        npd = NumpyDQN(W0)
        s, a, r, ns, term = [tf.constant(x) for x in batch]
        nq = tf.reduce_max(ag.target(ns), axis=1)
        y = r + ag.gamma * nq * (1.0 - term)
        with tf.GradientTape() as tape:
            qa = ag.q(s, training=True)
            qt = tf.gather_nd(qa, tf.stack([tf.range(BATCH), a], axis=1))
            loss = tf.reduce_mean(tf.square(y - qt))
        g_tf = [x.numpy() for x in tape.gradient(loss, ag.q.trainable_variables)]
        loss_np, g_np = npd.grads(batch)
        return {"loss_diff": abs(float(loss) - loss_np),
                "max_grad_diff": float(max(np.max(np.abs(a_ - b_)) for a_, b_ in zip(g_tf, g_np)))}
    safe(res, "check_numpy_vs_tf", check)

    # --- learning step, microseconds per UPDATE (batch 64)
    def tf_base():
        ag = TFBaseline(W0)
        return timeit(lambda: ag.learn(batch).numpy(), 400)
    safe(res, "learn_tf_keras_baseline_us", tf_base)

    def tf_man():
        ag = TFManual(W0, K=1)
        b = tuple(x[None] for x in batch)
        return timeit(lambda: ag.train(*b).numpy(), 400)
    safe(res, "learn_tf_manual_adam_us", tf_man)

    def tf_fused():
        K = 4
        ag = TFManual(W0, K=K)
        b = stack_batches(buf, K)
        return timeit(lambda: ag.train(*b).numpy(), 300) / K
    safe(res, "learn_tf_manual_fused4_per_update_us", tf_fused)

    def np_learn():
        ag = NumpyDQN(W0)
        return timeit(lambda: ag.learn(batch), 1500)
    safe(res, "learn_numpy_us", np_learn)

    # --- weight sync cost after an update (what act needs)
    def sync6():
        ag = TFBaseline(W0)
        ag.learn(batch)
        return timeit(lambda: [v.numpy() for v in ag.q.trainable_variables], 500)
    safe(res, "sync_6x_numpy_us", sync6)

    def sync1():
        ag = TFNpAct(W0)
        return timeit(lambda: ag.learn(batch), 400)
    safe(res, "learn_plus_single_transfer_sync_us", sync1)

    # --- acting
    def act_np():
        ag = NumpyDQN(W0)
        return timeit(lambda: ag.act(next(it), 0.0), 20000)
    safe(res, "act_numpy_us", act_np)

    def act_tf():
        ag = TFBaseline(W0)
        return timeit(lambda: ag.act(next(it), 0.0), 1500)
    safe(res, "act_tf_us", act_tf)

    # --- PyTorch (optional)
    def torch_bench():
        import torch
        torch.set_num_threads(1)
        net = torch.nn.Sequential(torch.nn.Linear(8, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64),
                                  torch.nn.ReLU(), torch.nn.Linear(64, 4))
        tgt = torch.nn.Sequential(torch.nn.Linear(8, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64),
                                  torch.nn.ReLU(), torch.nn.Linear(64, 4))
        tgt.load_state_dict(net.state_dict())
        opt = torch.optim.Adam(net.parameters(), lr=LR)
        S, A, R, NS, T = [torch.from_numpy(x) for x in batch]
        A = A.long()
        tp, pp = list(tgt.parameters()), list(net.parameters())

        def step():
            with torch.no_grad():
                y = R + GAMMA * tgt(NS).max(1).values * (1 - T)
            q = net(S).gather(1, A[:, None])[:, 0]
            loss = ((y - q) ** 2).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            with torch.no_grad():
                for t, p in zip(tp, pp):
                    t.mul_(1 - TAU).add_(p, alpha=TAU)
            return loss.item()
        res["learn_torch_us"] = timeit(step, 500)

        def act():
            with torch.no_grad():
                return int(net(torch.from_numpy(next(it))).argmax())
        res["act_torch_us"] = timeit(act, 3000)
    try:
        torch_bench()
    except Exception as e:
        res["torch_error"] = (type(e).__name__ + ": " + str(e))[:200]

    # --- JAX (optional)
    def jax_bench():
        import jax, jax.numpy as jnp
        P = [jnp.asarray(w) for w in W0]
        Tg = list(P); M = [jnp.zeros_like(w) for w in P]; V = [jnp.zeros_like(w) for w in P]

        def fwd(p, x):
            h = jax.nn.relu(x @ p[0] + p[1]); h = jax.nn.relu(h @ p[2] + p[3]); return h @ p[4] + p[5]

        def loss_fn(p, tg, s, a, r, ns, term):
            y = r + GAMMA * fwd(tg, ns).max(1) * (1 - term)
            q = jnp.take_along_axis(fwd(p, s), a[:, None], 1)[:, 0]
            return jnp.mean((jax.lax.stop_gradient(y) - q) ** 2)

        @jax.jit
        def step(p, tg, m, v, n, s, a, r, ns, term):
            loss, g = jax.value_and_grad(loss_fn)(p, tg, s, a, r, ns, term)
            n = n + 1.0
            lr_t = LR * jnp.sqrt(1 - 0.999 ** n) / (1 - 0.9 ** n)
            m = [0.9 * mi + 0.1 * gi for mi, gi in zip(m, g)]
            v = [0.999 * vi + 0.001 * gi * gi for vi, gi in zip(v, g)]
            p = [pi - lr_t * mi / (jnp.sqrt(vi) + 1e-7) for pi, mi, vi in zip(p, m, v)]
            tg = [TAU * pi + (1 - TAU) * ti for pi, ti in zip(p, tg)]
            return p, tg, m, v, n, loss
        state = [P, Tg, M, V, jnp.float32(0.0)]

        def run():
            p, tg, m, v, n, loss = step(*state, *batch)
            state[:] = [p, tg, m, v, n]
            return loss.block_until_ready()
        res["learn_jax_us"] = timeit(run, 500)
        res["jax_version"] = jax.__version__
    try:
        jax_bench()
    except Exception as e:
        res["jax_error"] = (type(e).__name__ + ": " + str(e))[:200]
    return res


# ===================== real training loop
def make_agent(variant):
    W0 = glorot(np.random.default_rng(0))
    if variant == "tf_baseline":
        return TFBaseline(W0)
    if variant == "tf_npact":
        return TFNpAct(W0)
    if variant == "numpy_all":
        return NumpyDQN(W0)
    raise ValueError(variant)


def run_loop(variant, eps, steps, timers, start_at=None):
    env = gym.make(ENV_ID)
    trans = collect_transitions(env, 6000)
    agent = make_agent(variant)
    buf = ReplayBuffer(100_000, 8, np.random.default_rng(1))
    for t in trans:
        buf.add(*t)
    for _ in range(20):
        agent.learn(buf.sample(BATCH)); agent.act(trans[0][0], 0.0)
    state, _ = env.reset(seed=1)
    acc = dict(act=0.0, env=0.0, buf_add=0.0, buf_sample=0.0, learn=0.0)
    n_ep = 0
    if start_at:
        while time.time() < start_at:
            time.sleep(0.01)
    t_start = PC()
    for step in range(1, steps + 1):
        if timers: t0 = PC()
        a = agent.act(state, eps)
        if timers: t1 = PC(); acc["act"] += t1 - t0
        ns, r, te, tr, _ = env.step(a)
        if te or tr:
            n_ep += 1
        if timers: t2 = PC(); acc["env"] += t2 - t1
        buf.add(state, a, r, ns, te)
        if timers: t3 = PC(); acc["buf_add"] += t3 - t2
        state = env.reset()[0] if (te or tr) else ns
        if timers and (te or tr): acc["env"] += PC() - t3; t3 = PC()
        if step % UPDATE_EVERY == 0:
            if timers: t5 = PC()
            batch = buf.sample(BATCH)
            if timers: t6 = PC(); acc["buf_sample"] += t6 - t5
            agent.learn(batch)
            if timers: acc["learn"] += PC() - t6
    total = PC() - t_start
    out = {"variant": variant, "eps": eps, "steps": steps, "timers": timers,
           "steps_per_s": steps / total, "episodes": n_ep}
    if timers:
        per = {k: v / steps * 1e6 for k, v in acc.items()}
        per["other"] = total / steps * 1e6 - sum(per.values())
        out["us_per_step"] = per
    return out


# ===================== orchestrator
def parse(text):
    out = []
    for line in text.splitlines():
        if line.startswith("RESULT "):
            _, tag, payload = line.split(" ", 2)
            out.append((tag, json.loads(payload)))
    return out


def worker(args_list):
    p = subprocess.run([sys.executable, os.path.abspath(__file__)] + args_list, capture_output=True, text=True)
    r = parse(p.stdout)
    if not r:
        print("worker failed:", args_list, "\n", p.stderr[-2500:])
    return r


def run_parallel(k, extra, delay=30):
    start_at = time.time() + delay
    cmd = [sys.executable, os.path.abspath(__file__), "loop"] + extra + ["--start-at", str(start_at)]
    procs = [subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) for _ in range(k)]
    rs = []
    for p in procs:
        so, _ = p.communicate()
        rs += [r for _, r in parse(so)]
    return rs


def orchestrate(steps):
    ncpu = os.cpu_count() or 1
    R = {"cpu_count": ncpu}
    print("[1/3] micro-benchmarks (learn / sync / act, plus torch & jax if installed) ...", flush=True)
    r = worker(["micro"])
    if not r:
        sys.exit(1)
    R["micro"] = r[0][1]
    print(f"[2/3] full training loops, {steps} steps each (eps=0.05, 1 thread) ...", flush=True)
    R["loops"] = {}
    for v in ("tf_baseline", "tf_npact", "numpy_all"):
        r = worker(["loop", "--variant", v, "--steps", str(steps), "--mode", "both"])
        R["loops"][v] = {t: x for t, x in r}
    print(f"[3/3] {ncpu} processes at once (single machine) ...", flush=True)
    R["parallel"] = []
    if ncpu > 1:
        for v in ("tf_npact", "numpy_all"):
            rs = run_parallel(ncpu, ["--variant", v, "--steps", str(steps), "--mode", "plain"])
            if rs:
                R["parallel"].append({"variant": v, "procs": ncpu,
                                      "per_proc": [round(x["steps_per_s"]) for x in rs],
                                      "total": round(sum(x["steps_per_s"] for x in rs))})
    with open("profile_learn_report.json", "w") as f:
        json.dump(R, f, indent=1)
    print_report(R)


def print_report(R):
    m = R["micro"]
    print("\n" + "=" * 70 + "\nCOPY EVERYTHING BETWEEN THE LINES AND SEND IT BACK\n" + "=" * 70)
    print(f"cpus={R['cpu_count']} | tensorflow {m['tf_version']} | gymnasium {m['gym_version']}")
    print("\n[A] correctness of the numpy DQN vs TF:", m.get("check_numpy_vs_tf", m.get("check_numpy_vs_tf_error")))
    print("\n[B] one learning update, batch 64 (microseconds)")
    base = m.get("learn_tf_keras_baseline_us")
    for k in ("learn_tf_keras_baseline_us", "learn_tf_manual_adam_us", "learn_tf_manual_fused4_per_update_us",
              "learn_numpy_us", "learn_torch_us", "learn_jax_us"):
        if k in m:
            sp = f"  (x{base / m[k]:.1f} vs baseline)" if base and k != "learn_tf_keras_baseline_us" else ""
            print(f"  {k:40s} {m[k]:9.1f}{sp}")
        elif k + "_error" in m:
            print(f"  {k:40s} ERROR {m[k + '_error']}")
    for k in ("torch_error", "jax_error"):
        if k in m:
            print(f"  {k}: {m[k]}")
    print("\n[C] weight sync / act (microseconds)")
    for k in ("sync_6x_numpy_us", "learn_plus_single_transfer_sync_us", "act_tf_us", "act_numpy_us", "act_torch_us"):
        if k in m:
            print(f"  {k:40s} {m[k]:9.1f}")
        elif k + "_error" in m:
            print(f"  {k:40s} ERROR {m[k + '_error']}")
    print("\n[D] full training loop (1 process, 1 thread, eps=0.05)")
    b = R["loops"].get("tf_baseline", {}).get("plain")
    for v, d in R["loops"].items():
        p, t = d.get("plain"), d.get("timed")
        if not p:
            continue
        sp = f"  x{p['steps_per_s'] / b['steps_per_s']:.2f} vs baseline" if b else ""
        mins = SWEEP_TOTAL_STEPS / p["steps_per_s"] / 60
        print(f"  {v:12s} {p['steps_per_s']:7.0f} steps/s{sp}   -> your sweep would take ~{mins:.1f} min")
        if t:
            print("      " + " | ".join(f"{k} {x:.0f}us" for k, x in t["us_per_step"].items()))
    print("\n[E] all cores at once")
    for d in R["parallel"]:
        print(f"  {d['variant']:12s} {d['procs']} procs: total {d['total']} steps/s   per-proc {d['per_proc']}")
    print("=" * 70 + "\nfull numbers also saved in profile_learn_report.json")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["all", "micro", "loop"])
    ap.add_argument("--steps", type=int, default=20000)
    ap.add_argument("--eps", type=float, default=0.05)
    ap.add_argument("--variant", default="tf_baseline", choices=["tf_baseline", "tf_npact", "numpy_all"])
    ap.add_argument("--mode", default="both", choices=["both", "timed", "plain"])
    ap.add_argument("--start-at", type=float, default=None)
    a = ap.parse_args()
    if a.cmd == "all":
        orchestrate(a.steps)
        return
    setup(need_tf=(a.cmd == "micro" or a.variant.startswith("tf")))
    if a.cmd == "micro":
        emit("micro", run_micro())
    else:
        if a.mode in ("both", "timed"):
            emit("timed", run_loop(a.variant, a.eps, a.steps, True))
        if a.mode in ("both", "plain"):
            emit("plain", run_loop(a.variant, a.eps, a.steps, False, a.start_at))


if __name__ == "__main__":
    main()
'''

open('profile_learn.py', 'w').write(SCRIPT)
p = subprocess.Popen([sys.executable, '-u', 'profile_learn.py', 'all'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
p.wait()

[1/3] micro-benchmarks (learn / sync / act, plus torch & jax if installed) ...
[2/3] full training loops, 20000 steps each (eps=0.05, 1 thread) ...
[3/3] 2 processes at once (single machine) ...

COPY EVERYTHING BETWEEN THE LINES AND SEND IT BACK
cpus=2 | tensorflow 2.21.0 | gymnasium 1.3.0

[A] correctness of the numpy DQN vs TF: {'loss_diff': 1.52587890625e-05, 'max_grad_diff': 1.1920928955078125e-07}

[B] one learning update, batch 64 (microseconds)
  learn_tf_keras_baseline_us                  2074.7
  learn_tf_manual_adam_us                     1554.4  (x1.3 vs baseline)
  learn_tf_manual_fused4_per_update_us         616.4  (x3.4 vs baseline)
  learn_numpy_us                               474.7  (x4.4 vs baseline)
  learn_torch_us                              1402.7  (x1.5 vs baseline)
  learn_jax_us                                 490.9  (x4.2 vs baseline)

[C] weight sync / act (microseconds)
  sync_6x_numpy_us                             517.9
  learn_plus_single_transfer_sync_

0

In [ ]:
import subprocess, sys, os, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swig"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium[box2d]"], check=True)

import numpy as np, pandas as pd
import multiprocessing as mp
import gymnasium as gym

# ------------------------------------------------------------------ settings (same as your gamma-sweep notebook)
GAMMAS     = [0.995, 0.99]
SEEDS      = [0, 1, 2]
MAX_EP     = 800          # episode cap per run
N_PROC     = 2            # parallel processes (Colab free = 2 vCPU)
ENV_ID     = "LunarLander-v3"
BUFFER, BATCH, UPDATE_EVERY = 100_000, 64, 4
EPS_DECAY, EPS_MIN = 0.995, 0.01
SOLVE, WINDOW, N_EVAL = 200.0, 100, 20
REFERENCE = {0.995: dict(ep=562, ev=187.8, succ=0.55), 0.99: dict(ep=722, ev=197.7, succ=0.50)}  # your TF run, seed 0

# ------------------------------------------------------------------ numpy DQN (gradients verified vs TF: diff ~1e-7)
GAMMA, LR, TAU = 0.99, 1e-3, 1e-3

def glorot(rng, shapes=((8,64),(64,64),(64,4))):
    W=[]
    for fi,fo in shapes:
        lim=np.sqrt(6.0/(fi+fo))
        W += [rng.uniform(-lim,lim,size=(fi,fo)).astype(np.float32), np.zeros(fo,np.float32)]
    return W

class NumpyDQN:
    """Whole DQN (8-64-64-4 ReLU, Adam, soft target update) in plain numpy. The acting weights ARE self.p."""
    def __init__(self, W0, gamma=GAMMA, seed=0):
        self.p=[w.copy() for w in W0]; self.t=[w.copy() for w in W0]
        self.m=[np.zeros_like(w) for w in W0]; self.v=[np.zeros_like(w) for w in W0]
        self.n=0; self.gamma=np.float32(gamma); self.rng=np.random.default_rng(seed)

    @staticmethod
    def _fwd(P, x):
        h1=np.maximum(x@P[0]+P[1],0.0); h2=np.maximum(h1@P[2]+P[3],0.0)
        return h1,h2,h2@P[4]+P[5]

    def act(self, s, eps):
        if eps>0.0 and self.rng.random()<eps: return int(self.rng.integers(4))
        return int(np.argmax(self._fwd(self.p, s)[2]))

    def grads(self, batch):
        s,a,r,ns,term=batch
        y = r + self.gamma*self._fwd(self.t, ns)[2].max(axis=1)*(1.0-term)
        h1,h2,q = self._fwd(self.p, s)
        B=s.shape[0]; idx=np.arange(B)
        err = q[idx,a]-y
        loss=float(np.mean(err*err))
        dq=np.zeros_like(q); dq[idx,a]=2.0*err/B
        dW3=h2.T@dq; db3=dq.sum(0)
        dh2=(dq@self.p[4].T)*(h2>0)
        dW2=h1.T@dh2; db2=dh2.sum(0)
        dh1=(dh2@self.p[2].T)*(h1>0)
        dW1=s.T@dh1; db1=dh1.sum(0)
        return loss,[dW1,db1,dW2,db2,dW3,db3]

    def learn(self, batch):
        loss,g=self.grads(batch)
        self.n+=1; b1,b2,eps=0.9,0.999,1e-7
        lr_t=LR*np.sqrt(1-b2**self.n)/(1-b1**self.n)
        for p,m,v,gr,t in zip(self.p,self.m,self.v,g,self.t):
            m*=b1; m+=(1-b1)*gr
            v*=b2; v+=(1-b2)*gr*gr
            p-=lr_t*m/(np.sqrt(v)+eps)
            t*=(1-TAU); t+=TAU*p
        return loss

class ReplayBuffer:
    def __init__(self, capacity, state_dim, rng):
        self.capacity, self.rng = capacity, rng
        self.states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int32)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_dim), dtype=np.float32)
        self.terminals = np.zeros(capacity, dtype=np.float32)
        self._next_idx, self.size = 0, 0
    def add(self, s, a, r, ns, term):
        i = self._next_idx
        self.states[i] = s; self.actions[i] = a; self.rewards[i] = r
        self.next_states[i] = ns; self.terminals[i] = float(term)
        self._next_idx = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
    def sample(self, n):
        idx = self.rng.integers(0, self.size, size=n)
        return (self.states[idx], self.actions[idx], self.rewards[idx], self.next_states[idx], self.terminals[idx])

# ------------------------------------------------------------------ one run
def train_run(job):
    gamma, seed = job
    try:
        from threadpoolctl import threadpool_limits
        threadpool_limits(1)                      # tiny matrices: BLAS threads only slow things down
    except Exception:
        pass
    name = f"gamma{gamma:g}_seed{seed}"
    env = gym.make(ENV_ID)
    agent = NumpyDQN(glorot(np.random.default_rng(seed)), gamma=gamma, seed=seed)
    buf = ReplayBuffer(BUFFER, 8, np.random.default_rng(seed + 1))
    eps, rewards, lengths, total_steps, solved_at = 1.0, [], [], 0, -1
    t0 = time.time()
    for ep in range(MAX_EP):
        state, _ = env.reset(seed=seed if ep == 0 else None)
        ep_r, ep_len, done = 0.0, 0, False
        while not done:
            a = agent.act(state, eps)
            ns, r, term, trunc, _ = env.step(a)
            buf.add(state, a, r, ns, term)        # truncation is NOT terminal (same as your notebook)
            state = ns; ep_r += r; ep_len += 1; total_steps += 1
            done = term or trunc
            if total_steps % UPDATE_EVERY == 0 and buf.size >= BATCH:
                agent.learn(buf.sample(BATCH))
        rewards.append(ep_r); lengths.append(ep_len)
        eps = max(EPS_MIN, eps * EPS_DECAY)
        avg = float(np.mean(rewards[-WINDOW:]))
        if (ep + 1) % 100 == 0:
            print(f"[{name}] ep {ep + 1:4d} | avg100 {avg:8.1f} | {(time.time() - t0) / 60:4.1f} min", flush=True)
        if len(rewards) >= WINDOW and avg >= SOLVE:
            solved_at = ep + 1
            break
    train_s = time.time() - t0
    ev = []
    for i in range(N_EVAL):                       # greedy evaluation of the final policy
        s, _ = env.reset(seed=10_000 + 100 * seed + i)
        d, tot = False, 0.0
        while not d:
            s, r, te, tr, _ = env.step(agent.act(s, 0.0))
            tot += r; d = te or tr
        ev.append(tot)
    ev = np.array(ev)
    print(f"[{name}] DONE: {'solved in %d ep' % solved_at if solved_at > 0 else 'NOT solved'} | "
          f"eval {ev.mean():.1f} | {train_s / 60:.1f} min", flush=True)
    return dict(gamma=gamma, seed=seed, solved=solved_at > 0, episodes=len(rewards),
                solved_at=solved_at if solved_at > 0 else np.nan,
                train_avg_last100=float(np.mean(rewards[-WINDOW:])),
                len_last100=float(np.mean(lengths[-WINDOW:])),
                eval_mean=float(ev.mean()), eval_std=float(ev.std()),
                eval_success=float(np.mean(ev >= SOLVE)),
                minutes=train_s / 60, steps=total_steps, steps_per_s=total_steps / train_s,
                rewards=np.array(rewards))

if __name__ == "__main__" or True:
    jobs = [(g, s) for s in SEEDS for g in GAMMAS]
    print(f"{len(jobs)} runs on {N_PROC} processes ...", flush=True)
    t_all = time.time()
    with mp.get_context("fork").Pool(N_PROC) as pool:
        results = list(pool.imap_unordered(train_run, jobs))
    wall = (time.time() - t_all) / 60

    df = pd.DataFrame([{k: v for k, v in r.items() if k != "rewards"} for r in results]) \
           .sort_values(["gamma", "seed"], ascending=[False, True]).reset_index(drop=True)
    np.savez_compressed("numpy_dqn_validation_rewards.npz",
                        **{f"gamma{r['gamma']:g}_seed{r['seed']}": r["rewards"] for r in results})
    df.to_csv("numpy_dqn_validation.csv", index=False)

    print("\n" + "=" * 70 + "\nCOPY EVERYTHING BETWEEN THE LINES AND SEND IT BACK\n" + "=" * 70)
    print(df.drop(columns=["steps"]).round({c: 2 for c in df.columns if c not in ("gamma", "steps")}).to_string(index=False))
    print("\nper gamma (numpy DQN, 3 seeds)  vs  your TF run (seed 0):")
    for g in GAMMAS:
        d = df[df.gamma == g]
        ref = REFERENCE[g]
        print(f"  gamma={g:g}: solved {int(d.solved.sum())}/{len(d)} | episodes to solve {d.solved_at.tolist()} "
              f"| eval mean {d.eval_mean.round(1).tolist()} | eval success {d.eval_success.tolist()}")
        print(f"             TF reference: solved at {ref['ep']} | eval {ref['ev']} | success {ref['succ']}")
    print(f"\nwall time {wall:.1f} min | mean throughput {df.steps_per_s.mean():.0f} steps/s per process")
    print("=" * 70)

6 runs on 2 processes ...
[gamma0.99_seed0] ep  100 | avg100   -145.4 |  0.2 min
[gamma0.995_seed0] ep  100 | avg100   -148.7 |  0.2 min
[gamma0.99_seed0] ep  200 | avg100    -86.7 |  0.4 min
[gamma0.995_seed0] ep  200 | avg100   -100.7 |  0.4 min
[gamma0.99_seed0] ep  300 | avg100    -48.2 |  1.0 min
[gamma0.995_seed0] ep  300 | avg100     -9.1 |  1.0 min
[gamma0.99_seed0] ep  400 | avg100     60.7 |  1.6 min
[gamma0.995_seed0] ep  400 | avg100    135.2 |  1.6 min
[gamma0.995_seed0] ep  500 | avg100    155.2 |  2.0 min
[gamma0.99_seed0] ep  500 | avg100    150.6 |  2.0 min
[gamma0.995_seed0] DONE: solved in 591 ep | eval 190.2 | 2.4 min
[gamma0.99_seed0] ep  600 | avg100    183.7 |  2.4 min
[gamma0.995_seed1] ep  100 | avg100   -149.7 |  0.1 min
[gamma0.99_seed0] DONE: solved in 636 ep | eval 154.3 | 2.6 min
[gamma0.99_seed1] ep  100 | avg100   -174.0 |  0.1 min
[gamma0.995_seed1] ep  200 | avg100    -73.2 |  0.3 min
[gamma0.99_seed1] ep  200 | avg100    -86.6 |  0.3 min
[gamma0.995_s

## Final analysis: where does DQN training time go, and what actually speeds it up?

**Question.** The gamma sweep in notebook 01 took 75.6 minutes on a 2-vCPU Colab CPU (3.75M environment steps, about 800 steps/s). Before running more experiments, where does that time go, which speed-ups are worth it, and does the fastest one still train the same agent?

**Approach.** (1) Profile the TensorFlow/Keras implementation component by component and in the real training loop. (2) Benchmark candidate speed-ups. (3) Re-train with the fastest candidate (a NumPy re-implementation) at gamma = 0.995 and 0.99, 3 seeds each, and compare with the TensorFlow results.

### 1. Where the time goes (TensorFlow baseline, 1 process, epsilon = 0.05)

| Component | Time per env step | Share |
|---|---|---|
| `act` (one `tf.function` call per step) | ~670 µs | 55% |
| `learn` (one update every 4 steps) | ~390 µs | 32% |
| Box2D `env.step` | ~130 µs | 11% |
| replay buffer + other | ~30 µs | 2% |

The profile reproduces the real sweep: predicted 75.7 min, measured 75.6 min. The cost is call overhead, not arithmetic. A forward pass of the 8-64-64-4 network takes about 650 µs through TensorFlow and about 15 µs in NumPy, with identical actions. A training step costs about the same at batch 64 and 256 (1.9 ms vs 1.6 ms) and only grows at batch 1024 (4.0 ms).

### 2. Candidate speed-ups

| Candidate | Result | Verdict |
|---|---|---|
| NumPy `act`, weights copied with 6 `.numpy()` calls | 1.58x (825 -> 1303 steps/s); the copy costs ~520 µs per update | partly wasted |
| NumPy `act`, weights copied in one transfer | 2.19x (792 -> 1732 steps/s) | good, no algorithm change |
| Whole DQN in NumPy (act, backprop, Adam, soft update) | **4.98x** (792 -> 3942 steps/s) | **best** |
| XLA (`jit_compile`) on the training step | slower (3.5 ms vs 1.9 ms) | rejected |
| Hand-written Adam in TensorFlow | 1.3x on `learn` | minor |
| 4 updates fused in one TF call | 3.4x on `learn`, but acting weights go stale | not adopted |
| PyTorch (CPU) | 1.5x on `learn` | not worth switching |
| JAX (CPU, jit) | 4.2x on `learn`, same as NumPy | no gain over NumPy |
| 2 processes, 1 thread each | +1.2x (NumPy) to +1.35x (TF + NumPy act) total | worth it, but below 2x |

Not measured: GPU, vectorised environments end to end, and PyTorch/JAX inside the full loop. The GPU is unlikely to help here (a ~5k-parameter network, batch 64, and Box2D stays on the CPU), but that is reasoning, not a measurement.

With the NumPy implementation, the 3.75M-step sweep would take about 13 minutes on 2 processes (4762 steps/s total) instead of 76. This is a projection from throughput, not a re-run of the sweep.

### 3. Does the NumPy agent still learn the same way?

The NumPy gradients match TensorFlow's to 1.2e-7 on identical weights and batch. Full training, same hyperparameters as notebook 01:

| gamma | Episodes to solve (NumPy, seeds 0/1/2) | Greedy eval mean | Eval success rate | TensorFlow, seed 0 |
|---|---|---|---|---|
| 0.995 | 591 / 525 / 560 | 190 / 208 / 248 | 70% / 60% / 100% | solved at 562, eval 188, 55% |
| 0.99 | 636 / 583 / 720 | 154 / 215 / 139 | 50% / 70% / 50% | solved at 722, eval 198, 50% |

All 6 runs solved the task. The TensorFlow results fall inside the NumPy seed ranges. The six runs took 7.7 minutes in total on 2 processes.

### 4. Conclusions

1. The training loop was overhead-bound: per-call cost of `tf.function` dominated, not computation. GPUs, bigger batches and XLA do not address that.
2. Replacing TensorFlow with a small NumPy DQN gives about 5x per process (about 6x with 2 processes) with no visible change in learning behaviour at gamma = 0.995 and 0.99. A TensorFlow-only fix (NumPy act + single weight transfer) gives 2.2x.
3. The largest remaining lever is experimental design, not code: 81% of the notebook 01 sweep's env steps went to runs that never solved (gamma <= 0.95, ~900-970 step episodes). A budget or early-stopping rule would save more than any further code optimisation, but it changes what is observed, so it should be a deliberate choice.
4. After the NumPy switch, time per step is roughly 30% Box2D, 55% learning and 10% acting; further gains from code are bounded (Box2D alone caps a single process at roughly 11k steps/s).

### 5. Limitations

- Colab CPUs are shared: the original sweep ran at 750-1150 steps/s depending on the run. Each benchmark ran once, so compare numbers only within the same session.
- Benchmarks use epsilon = 0.05 with a buffer pre-filled by random-policy data; the real epsilon schedule mixes cheaper early steps (at epsilon = 1 the TF loop ran at 1785 steps/s).
- The NumPy agent is not a bit-exact copy: Adam is hand-written (Keras formula, epsilon = 1e-7), weights use a hand-written Glorot initialisation, and the random streams differ. Validation is 3 seeds per gamma against one TensorFlow seed, so differences of about 100 episodes are within seed spread. Success rates from 20 evaluation episodes have a standard error of about 11 percentage points.
- NumPy was validated only at gamma = 0.995 and 0.99, not in the unsolved regime (gamma <= 0.95) where the agent hovers.
- The sub-linear 2-process scaling (about 1.2x for NumPy) is consistent with two vCPUs sharing physical cores, which was not checked.